# 03 — Model B: loan approval / risk tier

Dataset: `loan_approval_lk_synthetic.csv`  
Target: `loan_status` (Approved = 1)  
`risk_tier` (Poor / Fair / Good / Excellent) is **derived from CRIB bins**, not a second model.

Expect near-perfect F1 on this public file: approval is almost a CRIB threshold. That is a dataset property — keep CRIB in the model and discuss it in the report.


In [ ]:
%matplotlib inline


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "ml" / "pipeline" / "features.py").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.config import (
    DATA_RAW,
    FIGURES_DIR,
    MODELS_DIR,
    RANDOM_STATE,
    REPORTS_DIR,
    ensure_dirs,
)

ensure_dirs()
print("Project root:", ROOT)


In [ ]:
import json
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")


def ks_statistic(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    pos, neg = y_score[y_true == 1], y_score[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    return float(ks_2samp(pos, neg).statistic)


def classification_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (np.asarray(y_proba) >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "ks_statistic": ks_statistic(y_true, y_proba),
        "average_precision": float(average_precision_score(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(report["1"]["precision"]),
        "recall": float(report["1"]["recall"]),
        "classification_report": report,
    }


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": float(np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1, None))) * 100),
    }


def stratified_split(X, y, random_state=RANDOM_STATE):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def save_pipeline(pipe, name, metadata):
    path = MODELS_DIR / f"{name}.pkl"
    joblib.dump(pipe, path)
    metadata = {**metadata, "artifact": str(path.as_posix()), "saved_at": datetime.now(timezone.utc).isoformat()}
    (MODELS_DIR / f"{name}.meta.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
    print("Saved", path)
    return path


In [ ]:
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from ml.pipeline.features import LoanApprovalFeatures, strip_loan_frame

set_config(transform_output="pandas")

csv = DATA_RAW / "loan_approval_lk_synthetic.csv"
if not csv.exists():
    csv = ROOT / "loan_approval_lk_synthetic.csv"
loan_df = strip_loan_frame(pd.read_csv(csv))

y = (loan_df["loan_status"] == "Approved").astype(int)
X = loan_df.drop(columns=[c for c in ("loan_status", "loan_id") if c in loan_df.columns])
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(X, y)
scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f"train/val/test = {len(X_train)}/{len(X_val)}/{len(X_test)}  approved_rate={y_train.mean():.3f}")


## RandomForest vs XGBoost (same shared `LoanApprovalFeatures`)


In [ ]:
CAT_COLS = ["education", "self_employed", "risk_tier"]

def make_preprocessor(sample):
    cat = [c for c in CAT_COLS if c in sample.columns]
    num = [c for c in sample.columns if c not in cat]
    return ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat), ("num", "passthrough", num)],
        remainder="drop",
        verbose_feature_names_out=False,
    )

preview = LoanApprovalFeatures().fit(X_train).transform(X_train.head(5))

rf_pipe = Pipeline([
    ("features", LoanApprovalFeatures()),
    ("preprocess", make_preprocessor(preview)),
    ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
])
xgb_pipe = Pipeline([
    ("features", LoanApprovalFeatures()),
    ("preprocess", make_preprocessor(preview)),
    ("model", XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
        scale_pos_weight=scale_pos_weight, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE, eval_metric="logloss",
    )),
])

rf_pipe.fit(X_train, y_train)
xgb_pipe.fit(X_train, y_train)
rf_val = classification_metrics(y_val, rf_pipe.predict_proba(X_val)[:, 1])
xgb_val = classification_metrics(y_val, xgb_pipe.predict_proba(X_val)[:, 1])
print(f"RandomForest val F1={rf_val['f1']:.4f} AUC={rf_val['auc_roc']:.4f}")
print(f"XGBoost       val F1={xgb_val['f1']:.4f} AUC={xgb_val['auc_roc']:.4f}")

winner_name, pipe = ("random_forest", rf_pipe) if rf_val["f1"] >= xgb_val["f1"] else ("xgboost", xgb_pipe)
test_metrics = classification_metrics(y_test, pipe.predict_proba(X_test)[:, 1])
print(f"Winner={winner_name}  test F1={test_metrics['f1']:.4f}  AUC={test_metrics['auc_roc']:.4f}")
print(classification_report(y_test, pipe.predict(X_test), zero_division=0))


In [ ]:
proba = pipe.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, proba)
plt.figure(figsize=(6, 4.5))
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, proba):.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.title("ROC — Model B approval")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_b_roc.png", dpi=140)
plt.show()


In [ ]:
import shap

transformed = pipe.named_steps["preprocess"].transform(pipe.named_steps["features"].transform(X_test))
sample = transformed.sample(n=min(200, len(transformed)), random_state=1)
shap.summary_plot(shap.TreeExplainer(pipe.named_steps["model"]).shap_values(sample), sample, show=False, max_display=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary_model_b.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
# Derived grade-style output (not a trained 4-class model)
LoanApprovalFeatures().fit(X_test).transform(X_test)["risk_tier"].value_counts()


In [ ]:
save_pipeline(
    pipe,
    "model_b_approval",
    {
        "model_version": "model_b_v2_lk",
        "task": "loan_approval",
        "winner": winner_name,
        "metrics": {"val_rf_f1": rf_val["f1"], "val_xgb_f1": xgb_val["f1"], "test_f1": test_metrics["f1"], "test_auc": test_metrics["auc_roc"]},
        "notes": "Near-perfect F1: loan_status is almost a CRIB rule on this public dataset.",
    },
)
loaded = joblib.load(MODELS_DIR / "model_b_approval.pkl")
print("sample approve probability:", float(loaded.predict_proba(X_test.head(1))[:, 1][0]))
